# Project Title: [Your Project Name]

**Author:** [Your Name]  
**Date:** [Date]  
**Version:** 1.0

## Overview

Brief description of what this notebook does:
- What problem are you solving?
- What approach are you taking?
- What are the key findings?

---

## Table of Contents

1. [Setup](#1.-Setup)
2. [Data Loading](#2.-Data-Loading)
3. [Exploration](#3.-Exploration)
4. [Preprocessing](#4.-Preprocessing)
5. [Model Training](#5.-Model-Training)
6. [Evaluation](#6.-Evaluation)
7. [Conclusions](#7.-Conclusions)

---
## 1. Setup

Import libraries, set random seeds, configure settings.

In [ ]:
# Standard library imports
import os
import sys
import json
import time
from pathlib import Path

# Data science imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ML imports (uncomment as needed)
# import torch
# import torch.nn as nn
# from sklearn.model_selection import train_test_split

# Display settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("Setup complete!")

In [ ]:
# CRITICAL: Set random seeds for reproducibility
SEED = 42

np.random.seed(SEED)
# random.seed(SEED)  # If using Python's random module

# For PyTorch (uncomment if using)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)
#     torch.backends.cudnn.deterministic = True

print(f"Random seed set to: {SEED}")

In [ ]:
# Document versions for reproducibility
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
# print(f"PyTorch: {torch.__version__}")
# print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Configuration constants
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
MODEL_DIR = Path("models")

# Create directories if they don't exist
for d in [DATA_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(exist_ok=True)

# Hyperparameters
CONFIG = {
    'batch_size': 32,
    'learning_rate': 0.001,
    'epochs': 10,
    'hidden_dim': 256,
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

---
## 2. Data Loading

Load and inspect the data.

In [ ]:
# Example: Generate synthetic data for demonstration
# Replace with your actual data loading code

n_samples = 1000
n_features = 10

# Generate random data
X = np.random.randn(n_samples, n_features)
# Generate labels (binary classification)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
# Quick data inspection
print("Feature statistics:")
print(f"  Mean: {X.mean(axis=0).round(3)}")
print(f"  Std:  {X.std(axis=0).round(3)}")
print(f"  Min:  {X.min(axis=0).round(3)}")
print(f"  Max:  {X.max(axis=0).round(3)}")

---
## 3. Exploration

Visualize and understand the data.

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i in range(n_features):
    axes[i].hist(X[:, i], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Feature {i}')
    axes[i].set_xlabel('Value')

plt.tight_layout()
plt.suptitle('Feature Distributions', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Class separation visualization (first 2 features)
fig, ax = plt.subplots(figsize=(8, 6))

for label in [0, 1]:
    mask = y == label
    ax.scatter(X[mask, 0], X[mask, 1], label=f'Class {label}', alpha=0.6)

ax.set_xlabel('Feature 0')
ax.set_ylabel('Feature 1')
ax.set_title('Data Visualization (First 2 Features)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 4. Preprocessing

Prepare data for modeling.

In [ ]:
# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class dist: {np.bincount(y_train)}")
print(f"Test class dist:  {np.bincount(y_test)}")

In [ ]:
# Standardization (fit on train, transform both)
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std  # Use TRAIN stats!

print("After scaling (train):")
print(f"  Mean: {X_train_scaled.mean(axis=0).round(6)}")
print(f"  Std:  {X_train_scaled.std(axis=0).round(3)}")

---
## 5. Model Training

Train and tune the model.

In [ ]:
# Example: Simple logistic regression from scratch
# Replace with your actual model

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def train_logistic_regression(X, y, lr=0.01, epochs=100):
    """Train logistic regression with gradient descent."""
    n_samples, n_features = X.shape
    weights = np.zeros(n_features)
    bias = 0
    
    history = {'loss': [], 'accuracy': []}
    
    for epoch in range(epochs):
        # Forward pass
        z = X @ weights + bias
        predictions = sigmoid(z)
        
        # Compute loss (binary cross-entropy)
        loss = -np.mean(y * np.log(predictions + 1e-8) + 
                       (1 - y) * np.log(1 - predictions + 1e-8))
        
        # Compute accuracy
        accuracy = ((predictions > 0.5) == y).mean()
        
        history['loss'].append(loss)
        history['accuracy'].append(accuracy)
        
        # Backward pass
        error = predictions - y
        grad_weights = (1 / n_samples) * X.T @ error
        grad_bias = (1 / n_samples) * error.sum()
        
        # Update
        weights -= lr * grad_weights
        bias -= lr * grad_bias
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d}: loss={loss:.4f}, accuracy={accuracy:.4f}")
    
    return weights, bias, history

In [ ]:
# Train the model
print("Training...\n")
weights, bias, history = train_logistic_regression(
    X_train_scaled, y_train, 
    lr=CONFIG['learning_rate'], 
    epochs=CONFIG['epochs'] * 10
)
print("\nTraining complete!")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['accuracy'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Evaluation

Evaluate model performance on test set.

In [ ]:
# Predict on test set
def predict(X, weights, bias):
    return sigmoid(X @ weights + bias)

test_probs = predict(X_test_scaled, weights, bias)
test_preds = (test_probs > 0.5).astype(int)

# Calculate metrics
test_accuracy = (test_preds == y_test).mean()
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, test_preds)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Negative', 'Positive'])
ax.set_yticklabels(['Negative', 'Positive'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

# Add numbers
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, cm[i, j], ha='center', va='center', color=color, fontsize=16)

fig.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# Classification report
print("Classification Report:")
print(classification_report(y_test, test_preds, target_names=['Negative', 'Positive']))

---
## 7. Conclusions

### Summary

- **What we did:** Trained a logistic regression classifier on synthetic data
- **Key results:** Achieved X% accuracy on the test set
- **Limitations:** This is synthetic data; real-world data may be more complex

### Next Steps

1. Try more complex models (neural networks, ensemble methods)
2. Perform hyperparameter tuning
3. Add cross-validation
4. Apply to real dataset

In [ ]:
# Save results
results = {
    'test_accuracy': float(test_accuracy),
    'config': CONFIG,
    'seed': SEED,
}

with open(OUTPUT_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {OUTPUT_DIR / 'results.json'}")

---
## Appendix: Magic Commands Reference

Useful Jupyter magic commands for ML work:

In [ ]:
# Time a single line
%timeit np.dot(np.random.randn(100, 100), np.random.randn(100, 100))

In [ ]:
%%timeit
# Time an entire cell
result = 0
for i in range(1000):
    result += i

In [ ]:
# List all variables
%whos

In [ ]:
# Run shell commands
!echo "Hello from shell!"
# !pip list | head -10
# !nvidia-smi  # Check GPU

In [ ]:
# Auto-reload modules (useful during development)
# %load_ext autoreload
# %autoreload 2

---
## Best Practices Checklist

Before sharing this notebook:

- [ ] **Restart & Run All** - Verify cells run in order
- [ ] **Check random seeds** - Are results reproducible?
- [ ] **Document versions** - Python and key library versions logged
- [ ] **Clear outputs** (optional) - For smaller file size
- [ ] **Add markdown** - Explain each section
- [ ] **Remove debugging code** - Clean up temporary cells
- [ ] **Check data paths** - Are they relative, not absolute?